<a href="https://colab.research.google.com/github/nitindavegit/F1-Race-Predictions/blob/main/Australian_GP_2026_Predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module Installations

In [1]:
!pip install fastf1 xgboost pandas numpy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00


# Enabling fast caching

In [2]:
import os
import fastf1

os.makedirs('/content/cache', exist_ok=True)
fastf1.Cache.enable_cache('/content/cache')

# load aus gp dataset

In [3]:
year= 2026
race = "Australia"
# load FP2
fp2 = fastf1.get_session(year, race, 'FP2')
fp2.load()
# load Qualifying
quali = fastf1.get_session(year,race,'Q')
quali.load()

core           INFO 	Loading data for Australian Grand Prix - Practice 2 [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Practice 2 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found fo

# extract qualifying data

In [4]:
import pandas as pd
print(quali.results.columns)

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps'],
      dtype='object')


In [6]:
import pandas as pd

quali_results = quali.results[['Abbreviation','TeamName','Position','Q1','Q2','Q3']].copy()

quali_results.rename(columns={
     'Abbreviation': 'driver',
     'TeamName': 'team',
    'Position': 'grid_position'
}, inplace=True)

quali_results.head()

,driver,team,grid_position,Q1,Q2,Q3
63,RUS,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000
12,ANT,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000
6,HAD,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000
16,LEC,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000
81,PIA,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000


# Handling missing values in qualifying times

In [7]:
quali_results['qualifying_time'] = (
    quali_results[['Q3','Q2','Q1']]
    .bfill(axis=1)
    .iloc[:,0]
)

# Convert Lap Time to Seconds

In [8]:
quali_results['qualifying_seconds'] = (
    quali_results['qualifying_time'].dt.total_seconds()
)

# Handle Missing Qualifying Seconds

In [9]:
# Some drivers may have no lap (crash, etc).
max_quali = quali_results['qualifying_seconds'].max()

quali_results['qualifying_seconds'] = (
    quali_results['qualifying_seconds'].fillna(max_quali + 1)
)

# driver slower than everyone

# Compute gap from pole

In [10]:
pole_time = quali_results['qualifying_time'].min()
quali_results['gap_from_pole'] = (quali_results['qualifying_time'] - pole_time).dt.total_seconds()


In [11]:
quali_results.head()

,driver,team,grid_position,Q1,Q2,Q3,qualifying_time,qualifying_seconds,gap_from_pole
63,RUS,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000,0 days 00:01:18.518000,78.518,0.000
12,ANT,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000,0 days 00:01:18.811000,78.811,0.293
6,HAD,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000,0 days 00:01:19.303000,79.303,0.785
16,LEC,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000,0 days 00:01:19.327000,79.327,0.809
81,PIA,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000,0 days 00:01:19.380000,79.380,0.862


# Grid Probability Feature

In [12]:
quali_results['grid_probability'] = 1 / quali_results['grid_position']

# Team performance score

In [13]:
team_points_2025 = {
    "Red Bull Racing": 451,
    "Ferrari": 398,
    "Mercedes": 469,
    "McLaren": 833,
    "Aston Martin": 89,
    "Alpine": 22,
    "Racing Bulls": 92,
    "Williams": 137,
    "Kick Sauber": 70,
    "Haas F1 Team": 79
}

max_points = max(team_points_2025.values())

quali_results['team_score'] = quali_results['team'].map(lambda x: team_points_2025.get(x,50) / max_points)

In [14]:
quali_results.head()

,driver,team,grid_position,Q1,Q2,Q3,qualifying_time,qualifying_seconds,gap_from_pole,grid_probability,team_score
63,RUS,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000,0 days 00:01:18.518000,78.518,0.000,1.000000,0.563025
12,ANT,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000,0 days 00:01:18.811000,78.811,0.293,0.500000,0.563025
6,HAD,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000,0 days 00:01:19.303000,79.303,0.785,0.333333,0.541417
16,LEC,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000,0 days 00:01:19.327000,79.327,0.809,0.250000,0.477791
81,PIA,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000,0 days 00:01:19.380000,79.380,0.862,0.200000,1.000000


# Add weather data

In [15]:
quali_results['rain_probability'] = 0.05
quali_results['temperature'] = 19

In [16]:
quali_results.head()

,driver,team,grid_position,Q1,Q2,Q3,qualifying_time,qualifying_seconds,gap_from_pole,grid_probability,team_score,rain_probability,temperature
63,RUS,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000,0 days 00:01:18.518000,78.518,0.000,1.000000,0.563025,0.05,19
12,ANT,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000,0 days 00:01:18.811000,78.811,0.293,0.500000,0.563025,0.05,19
6,HAD,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000,0 days 00:01:19.303000,79.303,0.785,0.333333,0.541417,0.05,19
16,LEC,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000,0 days 00:01:19.327000,79.327,0.809,0.250000,0.477791,0.05,19
81,PIA,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000,0 days 00:01:19.380000,79.380,0.862,0.200000,1.000000,0.05,19


# Extract FP2 Lap Data

In [17]:
fp2_laps = fp2.laps
fp2_laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:14:55.294000,NOR,1,NaT,1.0,1.0,0 days 00:12:59.921000,NaT,NaT,0 days 00:00:23.001000,...,True,McLaren,0 days 00:12:59.921000,2026-03-06 05:00:12.101,1,NaN,False,,False,False
1,0 days 00:16:21.246000,NOR,1,0 days 00:01:25.952000,2.0,1.0,NaT,NaT,0 days 00:00:29.509000,0 days 00:00:18.380000,...,True,McLaren,0 days 00:14:55.294000,2026-03-06 05:02:07.474,12,NaN,False,,False,True
2,0 days 00:17:59.501000,NOR,1,0 days 00:01:38.255000,3.0,1.0,NaT,NaT,0 days 00:00:32.037000,0 days 00:00:19.145000,...,True,McLaren,0 days 00:16:21.246000,2026-03-06 05:03:33.426,1,NaN,False,,False,True
3,0 days 00:27:25.498000,NOR,1,NaT,4.0,1.0,NaT,0 days 00:20:17.780000,0 days 00:00:52.162000,0 days 00:00:32.327000,...,True,McLaren,0 days 00:17:59.501000,2026-03-06 05:05:11.681,1,NaN,False,,False,False
4,0 days 00:29:02.397000,NOR,1,0 days 00:01:36.899000,5.0,2.0,0 days 00:27:27.972000,NaT,0 days 00:00:36.873000,0 days 00:00:18.865000,...,False,McLaren,0 days 00:27:25.498000,2026-03-06 05:14:37.678,1,NaN,False,,False,False


#Filter Valid Race Pace Laps

In [18]:
fp2_laps = fp2_laps.pick_quicklaps()

fp2_laps = fp2_laps[fp2_laps['LapTime'].notnull()]

# Convert Lap Time to Seconds

In [19]:
fp2_laps['lap_seconds'] = fp2_laps['LapTime'].dt.total_seconds()

# Compute Average Race Pace per Driver

In [20]:
fp2_pace = fp2_laps.groupby('Driver')['lap_seconds'].mean().reset_index()

fp2_pace.rename(columns = {
    'Driver': 'driver',
    'lap_seconds': 'fp2_avg_pace'
}, inplace=True)

fp2_pace.head()

,driver,fp2_avg_pace
0,ALB,83.281429
1,ALO,84.800500
2,ANT,83.283312
3,BEA,82.861800
4,BOR,82.146500


# Merge FP2 Pace With Qualifying Data

In [21]:
dataset = pd.merge(
    quali_results,
    fp2_pace,
    on='driver',
    how='left'
)

dataset.head()

,driver,team,grid_position,Q1,Q2,Q3,qualifying_time,qualifying_seconds,gap_from_pole,grid_probability,team_score,rain_probability,temperature,fp2_avg_pace
0,RUS,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000,0 days 00:01:18.518000,78.518,0.000,1.000000,0.563025,0.05,19,83.084643
1,ANT,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000,0 days 00:01:18.811000,78.811,0.293,0.500000,0.563025,0.05,19,83.283312
2,HAD,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000,0 days 00:01:19.303000,79.303,0.785,0.333333,0.541417,0.05,19,83.146636
3,LEC,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000,0 days 00:01:19.327000,79.327,0.809,0.250000,0.477791,0.05,19,82.152778
4,PIA,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000,0 days 00:01:19.380000,79.380,0.862,0.200000,1.000000,0.05,19,82.921364


# Fill Missing FP2 Pace

In [22]:
dataset['fp2_avg_pace'] = dataset['fp2_avg_pace'].fillna(
    dataset['fp2_avg_pace'].mean()
)

# FP2 delta feature

In [23]:
dataset['fp2_delta'] =  dataset['fp2_avg_pace'] - dataset['fp2_avg_pace'].min()

# Load Race Results

In [24]:
actual_race = fastf1.get_session(year, race, 'R')
actual_race.load()

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_st

# Extract Finishing Positions

In [25]:
race_results = actual_race.results[['Abbreviation','Position']].copy()

race_results.rename(columns = {
    'Abbreviation': 'driver',
    'Position': 'finish_position'
}, inplace=True)

race_results.head()

,driver,finish_position
63,RUS,1.0
12,ANT,2.0
16,LEC,3.0
44,HAM,4.0
1,NOR,5.0


# Merge Race Results With Dataset

In [26]:
dataset = pd.merge(
     dataset,
    race_results,
    on='driver',
    how='left'
)
dataset.head()

,driver,team,grid_position,Q1,Q2,Q3,qualifying_time,qualifying_seconds,gap_from_pole,grid_probability,team_score,rain_probability,temperature,fp2_avg_pace,fp2_delta,finish_position
0,RUS,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000,0 days 00:01:18.518000,78.518,0.000,1.000000,0.563025,0.05,19,83.084643,1.376643,1.0
1,ANT,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000,0 days 00:01:18.811000,78.811,0.293,0.500000,0.563025,0.05,19,83.283312,1.575312,2.0
2,HAD,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000,0 days 00:01:19.303000,79.303,0.785,0.333333,0.541417,0.05,19,83.146636,1.438636,20.0
3,LEC,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000,0 days 00:01:19.327000,79.327,0.809,0.250000,0.477791,0.05,19,82.152778,0.444778,3.0
4,PIA,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000,0 days 00:01:19.380000,79.380,0.862,0.200000,1.000000,0.05,19,82.921364,1.213364,21.0


# Prepare Model Inputs

In [27]:
feature_cols = [
    'grid_position',
    'gap_from_pole',
    'grid_probability',
    'team_score',
    'fp2_delta',
    'rain_probability',
    'temperature'
]

In [28]:
dataset.isna().sum()

,0
driver,0
team,0
grid_position,3
Q1,3
Q2,6
Q3,13
qualifying_time,3
qualifying_seconds,0
gap_from_pole,3
grid_probability,3


In [29]:
# Prepare Training Data
X = dataset[feature_cols]
y = dataset['finish_position']

# Train Model

In [30]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4
)

model.fit(X,y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

# Predictions

In [31]:
dataset['predicted_finish'] = model.predict(X)

# Predicted Podium

In [37]:
predicted_podium = dataset.sort_values('predicted_finish')[['driver','predicted_finish']].head(3)

# Create comparison table

In [38]:
comparison = dataset[['driver','finish_position','predicted_finish']]

comparison = comparison.sort_values('finish_position')

comparison

,driver,finish_position,predicted_finish
0,RUS,1.0,1.031515
1,ANT,2.0,2.025167
3,LEC,3.0,3.110166
6,HAM,4.0,4.006212
5,NOR,5.0,5.106936
20,VER,6.0,6.045142
11,BEA,7.0,7.013034
8,LIN,8.0,8.052720
9,BOR,9.0,9.012947
13,GAS,10.0,10.015585


# Actual Podium

In [39]:
actual_podium = dataset.sort_values('finish_position')[['driver']].head(3)

(actual_podium)

,driver
0,RUS
1,ANT
3,LEC


# Podium Accuracy

In [40]:
pred = set(predicted_podium['driver'])
actual = set(actual_podium['driver'])

correct = len(pred.intersection(actual))

print("Correct podium drivers:", correct, "/ 3")

Correct podium drivers: 3 / 3


# MAE

In [41]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    dataset['finish_position'],
    dataset['predicted_finish']
)

print("Mean Absolute Error:", mae)

Mean Absolute Error: 0.055643471804532135
